# 11.06 · Context engineering for long-running agents

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1.5–2 h |
| 📋 Prerequisites | Lessons 11.03–11.05; Lesson 03.05 (context engineering); Lesson 03.03 (long conversations). |
| 🔑 Keys | none for 🧪 cells · optional 💸 cell ≈ 10–15K tokens (off by default) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** the three context levers of the harness: offloading large tool results, summarizing old history, and isolating work in subagents.
2. **See** an offload happen, and read the offloaded result back in pages.
3. **Configure** summarization (trigger, keep, summary model) and know what the model sees vs what state keeps.
4. **Budget** a call's context: system prompt, tool schemas, history.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Context engineering in Deep Agents](https://docs.langchain.com/oss/python/deepagents/context-engineering) | Offloading, summarization, subagent isolation |
| [Built-in middleware: summarization](https://docs.langchain.com/oss/python/langchain/middleware/built-in) | `trigger`, `keep`, the summary model |
| [Changelog: v0.4.0](https://docs.langchain.com/oss/python/deepagents/changelog-py) | Summarization in the model node; full history kept in state; `ContextOverflowError` trigger |
| [Context engineering (LangChain)](https://docs.langchain.com/oss/python/langchain/context-engineering) | The general picture from Module 03 |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-context'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Helpers (same as Lesson 11.05) + a deliberately huge search tool.
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage

from mastery.deep import build_deep_agent, run_stats
from mastery.testing import ScriptedChatModel


def calls(*items) -> AIMessage:
    return AIMessage("", tool_calls=[{"name": n, "args": a, "id": i} for n, a, i in items])


def user(text: str) -> dict:
    return {"messages": [{"role": "user", "content": text}]}


@tool
def wide_search(query: str) -> str:
    """Search the web and return every result in full (a badly designed tool, on purpose)."""
    return "\n".join(f"Result {i}: {query} — page text page text page text page text page text." for i in range(600))


print("wide_search returns ≈", len(wide_search.invoke({"query": "agents"})) // 4, "tokens")

---
## Part 1 · Three levers

| Lever | Problem it solves | Built-in |
|---|---|---|
| **Offload** | One tool result floods the context | `FilesystemMiddleware`: results over `tool_token_limit_before_evict` (default 20,000 tokens) go to `/large_tool_results/<id>` |
| **Summarize** | Many turns slowly fill the context | `SummarizationMiddleware`: at the trigger, older turns are replaced by a summary for the model; the full text is saved to `/conversation_history/` |
| **Isolate** | Sub-tasks pollute the main thread with noise | Subagents (Lesson 11.05): only their final message comes back |

## Part 2 · Offloading in action

In [ ]:
# 🧪 Large tool results are OFFLOADED to a file; the model gets a pointer instead of the text.
from deepagents import FilesystemMiddleware

script = ScriptedChatModel(responses=[
    calls(("wide_search", {"query": "agent memory"}, "w1")),
    calls(("read_file", {"file_path": "/large_tool_results/w1", "offset": 0, "limit": 3}, "r1")),
    AIMessage("Read the first lines only.")])
# Default threshold: 20,000 tokens per tool result. We lower it to 2,000 so the demo stays small.
agent = build_deep_agent(script, tools=[wide_search], fallbacks=False,
                         middleware=[FilesystemMiddleware(tool_token_limit_before_evict=2_000)])
out = agent.invoke(user("Research agent memory."))
tool_texts = [m.content for m in out["messages"] if m.type == "tool"]
print("what the model saw for the search:\n ", tool_texts[0][:230].replace("\n", "\n  "), "…")
print("\nthen read_file (3 lines):\n ", tool_texts[1].replace("\n", "\n  ")[:300])
print("\nfiles:", list(out["files"]))

**You should now see** a short notice instead of ≈ 10K tokens of results ("Tool result too large … saved in the filesystem at this path: /large_tool_results/w1"), a paged `read_file` with only 3 lines, and the file in `files`. The model now pays for what it reads, not for everything the tool returned. Project 11 requirement 5 asks you to show one offload in a trace: this is what it looks like.

> ⚠️ **Production note: fix the tool before you rely on offloading.** Offloading is a safety net. In production: tools return the top-k, truncated and with IDs (Lesson 05.01 capped results to fit Groq's 8K TPM); offloading then catches the rare outlier.

## Part 3 · Summarization

In [ ]:
# 🧪 Summarization: when the history grows past a trigger, older turns become a summary for the MODEL.
from deepagents.backends import StateBackend
from deepagents.middleware.summarization import SummarizationMiddleware

summarizer = ScriptedChatModel(responses=[AIMessage("## SESSION INTENT\nPlan a team offsite.\n## SUMMARY\nBudget 2,000 EUR; "
                                                     "venue Lisbon chosen; dates 12-14 May.")], repeat_last=True)
main = ScriptedChatModel(responses=[AIMessage("Booked flights within the 2,000 EUR budget.")], repeat_last=True)
# Same name as the built-in -> replaces it (v0.7). Tiny trigger so the demo fires: 6 messages, keep the last 2.
compactor = SummarizationMiddleware(model=summarizer, backend=StateBackend(), trigger=("messages", 6), keep=("messages", 2))

history = []
for i, (q, a) in enumerate([("Budget?", "2,000 EUR."), ("Where?", "Lisbon."), ("When?", "12-14 May."),
                            ("Hotel?", "Near the river."), ("Food?", "Vegetarian options.")]):
    history += [HumanMessage(q), AIMessage(a)]
out = build_deep_agent(main, middleware=[compactor], fallbacks=False).invoke(
    {"messages": history + [HumanMessage("Book the flights.")]})
seen = main.calls[-1]
print("messages in state      :", len(out["messages"]), "(full history kept)")
print("messages the model saw :", len(seen))
for m in seen[1:]:
    print(f"  {m.type:<6} {m.text[:90]!r}")
print("history saved to       :", [p for p in out["files"] if p.startswith("/conversation_history/")])

**You should now see** 12 messages in state but only 4 sent to the model: the system prompt, a summary message ("You are in the middle of a conversation that has been summarized…" with the summarizer's text), the last kept turn, and the new request. The full transcript went to `/conversation_history/…md`, so the agent can `read_file` it if a detail was lost.

Since v0.4, summarization happens inside the model call: **state keeps the full history** (your checkpointer, your traces), only the model's view is compacted. It also fires automatically if a provider raises `ContextOverflowError`.

In [ ]:
# 🧪 What triggers summarization by default? It depends on the model's profile (max_input_tokens).
from deepagents.middleware.summarization import compute_summarization_defaults

from mastery.deep import deep_model

for spec in (DEEP_MODEL, "groq:openai/gpt-oss-20b"):
    model = deep_model(spec=spec, api_key="offline-placeholder")      # constructing a model makes no call
    window = (model.profile or {}).get("max_input_tokens")
    defaults = compute_summarization_defaults(model)
    print(f"{spec:<32} window={window} trigger={defaults['trigger']} keep={defaults['keep']}")
print(f"{'(no profile)':<32} ->", compute_summarization_defaults(ScriptedChatModel(responses=[AIMessage('x')]))["trigger"])

**What just happened?** Defaults follow the model's window: with a profile, the trigger is 85% of `max_input_tokens` and 10% is kept; without one, 170,000 tokens and the last 6 messages. On Gemini's 1M window, summarization will almost never fire in a course-sized run; on a 131K Groq model it fires sooner. On a free tier the binding limit is usually **tokens per minute or per day**, long before the window: that's why offloading and isolation matter more than summarization here.

> ⚠️ **Production note: summaries lose details.** "Context lost after summarization" is a top long-run failure (Lesson 11.11). In production: keep critical facts in **files** (`/research/notes.md`), not only in the chat; use a strong, cheap model for summaries; and test runs that cross the trigger.

## Part 4 · A context budget

In [ ]:
# 🧪 Context budget of one call: where do the tokens go? (≈ chars / 4)
def context_budget(system: str, tool_schemas: int, messages: list, window: int) -> dict:
    used = len(system) // 4 + tool_schemas + sum(len(str(m.content)) for m in messages) // 4
    return {"used≈tok": used, "window": window, "fill": f"{used / window:.1%}"}


table([{"case": "after offload", **context_budget("x" * 400, 2_300, out["messages"][-4:], 131_072)},
       {"case": "raw 600 results", **context_budget("x" * 400, 2_300, [AIMessage(wide_search.invoke({"query": "q"}))], 131_072)}],
      ["case", "used≈tok", "window", "fill"])

**What just happened?** The same conversation with the offloaded pointer is tiny; with the raw result inlined it costs ≈ 12K tokens **on every following call** (≈ 10% of a 131K window), which alone exceeds Groq's 8,000 tokens/minute free-tier limit.

## Part 5 · 💸 A real model and an offloaded result

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE (opt-in): a real model meets an offloaded result. Does it page through it sensibly? ≈ 10–15K tokens.
RUN_LIVE = False
if RUN_LIVE and HAVE_DEEP_KEY:
    live = build_deep_agent(tools=[wide_search], model_calls=8, tool_calls=8,
                            middleware=[FilesystemMiddleware(tool_token_limit_before_evict=2_000)])
    result = live.invoke(user("Use wide_search for 'agent memory', then tell me what result 5 says."))
    print("tools used:", [m.name for m in result["messages"] if m.type == "tool"])
    print("answer    :", result["messages"][-1].text[:300])
    print("stats     :", run_stats(result))
elif not RUN_LIVE:
    print("⏭️ Skipped: set RUN_LIVE = True to try it (≈ 10–15K tokens).")
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** (when enabled) `wide_search`, then one or more `read_file` calls with offsets, and an answer about result 5. If the model reads the whole file in one go, that's a context-engineering failure to note.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| 429 TPM right after a search | A big tool result re-sent on every call | Lower `tool_token_limit_before_evict`; make the tool return less |
| Agent "forgets" an early decision | It was summarized away | Write decisions to a file; re-read it |
| Offloaded files appear on disk | `FilesystemBackend` as the default backend | Keep `StateBackend` default; route disk under a prefix (Lesson 11.04) |
| Summarization never fires | Huge window (Gemini: 1M) | Expected; or set a smaller `trigger` deliberately |
| Two summarization middlewares | Added one with a different name | Use `SummarizationMiddleware` (same name replaces the built-in) |

## 🏋️ Exercises

1. **(Easy)** Set `tool_token_limit_before_evict=None` in Part 2 and look at the first tool message. What changed?
2. **(Medium)** Change the summarization trigger to `("tokens", 200)` and keep `("messages", 4)`. How many messages does the model see now?
3. **(Medium)** Rewrite `wide_search` to return the top 5 results with IDs and a `more_available` flag. Does it still get offloaded?
4. **(Stretch)** Add `SummarizationToolMiddleware(compactor)` (`deepagents.middleware.summarization`) and script a `compact_conversation` call: the agent compacts on demand (the tool answers "Conversation compacted…").

<details>
<summary><b>Show solutions</b></summary>

**1.** No offloading: the whole result lands in the message (`None` disables the limit). Don't do this on a free tier.

**2.** The trigger counts approximate tokens of the history, so it fires right away; the model sees 6 messages: the system prompt, the summary and the last 4 (including your new request).

**3.** No: 5 short results are far below 2,000 tokens.
```python
@tool
def top_search(query: str) -> dict:
    "Search the web; return the top 5 results with IDs."
    return {"results": [{"id": f"r{i}", "text": f"{query} result {i}"} for i in range(5)], "more_available": True}
```

**4.**
```python
from deepagents.middleware.summarization import SummarizationToolMiddleware
s = ScriptedChatModel(responses=[calls(("compact_conversation", {}, "c1")), AIMessage("compacted")])
build_deep_agent(s, middleware=[compactor, SummarizationToolMiddleware(compactor)], fallbacks=False).invoke(
    {"messages": history + [HumanMessage("Tidy up, then continue.")]})
```
</details>

## 🔎 Docs drill (5 min)

Where does a deep agent store the full text of messages it summarized away, and does the graph state still contain them?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Read the module docstring: `??deepagents.middleware.summarization` (Storage section), and the [v0.4.0 changelog](https://docs.langchain.com/oss/python/deepagents/changelog-py).
2. The evicted messages are written as Markdown to `/conversation_history/{session_id}.md` on the backend.
3. Yes: since v0.4 summarization happens in the model call and the full history stays in state.

</details>

## ✅ Checkpoint

- [ ] I can name the three context levers and which built-in implements each.
- [ ] I saw an offload and a paged read, and I know the default threshold.
- [ ] I can configure summarization and explain what the model sees vs what state keeps.
- [ ] I keep critical facts in files, not only in the chat.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

On [Context engineering in Deep Agents](https://docs.langchain.com/oss/python/deepagents/context-engineering), find the current offload threshold and the default summarization settings. Do they match what this notebook printed for `deepagents` 0.7.26? Log differences with versions in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Offload big tool results, summarize old turns, isolate sub-tasks.
- Offloaded results live in `/large_tool_results/`; summarized history in `/conversation_history/`; state keeps everything.
- Summarization defaults follow the model's window; on free tiers, per-minute and per-day token limits bind first.
- Write decisions to files so summaries can't lose them.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-07-skills-and-memory.ipynb** → [`11-07-skills-and-memory.ipynb`](11-07-skills-and-memory.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)